In [1]:
import time
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms


In [2]:
SEED = 42
EPOCHS = 5
BATCH_SIZE = 32
NUM_CLASSES = 10
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [3]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [4]:
# Same images, but training uses train_transform and validation uses eval_transform
train_full = datasets.CIFAR10("data", train=True, download=True, transform=train_transform)
val_full = datasets.CIFAR10("data", train=True, download=True, transform=eval_transform)

# First 1000 images for training, next 500 for validation (no overlap)
train_data = Subset(train_full, range(0, 1000))
val_data = Subset(val_full, range(1000, 1500))

print("Training images:", len(train_data))
print("Validation images:", len(val_data))
print("Classes:", NUM_CLASSES)

Training images: 1000
Validation images: 500
Classes: 10


In [5]:
raw_data = datasets.CIFAR10("data", train=True, download=True)  # no transform = raw image
img, label = raw_data[0]

train_outputs = [train_transform(img) for _ in range(5)]
eval_outputs = [eval_transform(img) for _ in range(5)]

all_different = True
for i in range(5):
    for j in range(i + 1, 5):
        if torch.equal(train_outputs[i], train_outputs[j]):
            all_different = False
print("5 train_transform outputs all different:", all_different)

all_same = True
for i in range(1, 5):
    if not torch.equal(eval_outputs[0], eval_outputs[i]):
        all_same = False
print("5 eval_transform outputs all identical:", all_same)

5 train_transform outputs all different: True
5 eval_transform outputs all identical: True


In [6]:
def build_model():
    set_seed(SEED)  # same seed = same starting head in both runs
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)  # replace the head
    return model

def count_trainable(model):
    total = 0
    for p in model.parameters():
        if p.requires_grad:
            total += p.numel()
    return total

def get_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += len(y)
    return correct / total

In [7]:
set_seed(SEED)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

model_fe = build_model()

# Freeze every parameter, then unfreeze only the new head
for param in model_fe.parameters():
    param.requires_grad = False
for param in model_fe.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model_fe.fc.parameters(), lr=LR_HEAD)
loss_fn = nn.CrossEntropyLoss()

start = time.time()
for epoch in range(EPOCHS):
    model_fe.eval()  # keep frozen backbone in eval() so BatchNorm stats don't change
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model_fe(x), y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}/{EPOCHS} - last batch loss: {loss.item():.3f}")
fe_time = time.time() - start

fe_acc = get_accuracy(model_fe, val_loader)
fe_params = count_trainable(model_fe)
print(f"Feature extraction -> acc: {fe_acc:.3f}, time: {fe_time:.1f}s, params: {fe_params}")

Epoch 1/5 - last batch loss: 2.000
Epoch 2/5 - last batch loss: 1.726
Epoch 3/5 - last batch loss: 1.464
Epoch 4/5 - last batch loss: 1.496
Epoch 5/5 - last batch loss: 1.438
Feature extraction -> acc: 0.760, time: 138.2s, params: 5130


In [8]:
set_seed(SEED)  # same seed as run 1 so the shuffling is the same too
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

model_ft = build_model()

# Freeze everything, then unfreeze layer4 (last block) and the head
for param in model_ft.parameters():
    param.requires_grad = False
for param in model_ft.layer4.parameters():
    param.requires_grad = True
for param in model_ft.fc.parameters():
    param.requires_grad = True

# ONE optimizer with two parameter groups
optimizer = torch.optim.AdamW([
    {"params": model_ft.fc.parameters(), "lr": LR_HEAD},
    {"params": model_ft.layer4.parameters(), "lr": LR_BACKBONE},
])
loss_fn = nn.CrossEntropyLoss()

start = time.time()
for epoch in range(EPOCHS):
    model_ft.eval()         # frozen layers stay in eval()
    model_ft.layer4.train() # only the unfrozen block trains normally
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model_ft(x), y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}/{EPOCHS} - last batch loss: {loss.item():.3f}")
ft_time = time.time() - start

ft_acc = get_accuracy(model_ft, val_loader)
ft_params = count_trainable(model_ft)
print(f"Fine-tuning -> acc: {ft_acc:.3f}, time: {ft_time:.1f}s, params: {ft_params}")

Epoch 1/5 - last batch loss: 1.401
Epoch 2/5 - last batch loss: 1.144
Epoch 3/5 - last batch loss: 1.124
Epoch 4/5 - last batch loss: 0.820
Epoch 5/5 - last batch loss: 1.274
Fine-tuning -> acc: 0.854, time: 135.8s, params: 8398858


In [9]:
print(f"{'Approach':20} | {'val acc':>8} | {'train time':>10} | {'trainable params':>16}")
print("-" * 65)
print(f"{'Feature extraction':20} | {fe_acc:>8.3f} | {fe_time:>9.1f}s | {fe_params:>16,}")
print(f"{'Fine-tuning':20} | {ft_acc:>8.3f} | {ft_time:>9.1f}s | {ft_params:>16,}")

Approach             |  val acc | train time | trainable params
-----------------------------------------------------------------
Feature extraction   |    0.760 |     138.2s |            5,130
Fine-tuning          |    0.854 |     135.8s |        8,398,858


In [10]:
if ft_acc > fe_acc:
    print("Fine-tuning won.")
elif ft_acc < fe_acc:
    print("Feature extraction won.")
else:
    print("They tied.")

Fine-tuning won.
